# Parameter diagnostics: variational dispersion + library centering experiments

Compare trained model parameters across 7 experiments:
- **Exp 1-4**: Single-modal RNA (regularise_bg x centering)
- **Exp 5-7**: Multimodal RNA+ATAC (centering variations)

All experiments use variational LogNormal dispersion posterior.

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams["pdf.fonttype"] = 42
rcParams["figure.figsize"] = 12, 4

## 1. Experiment configuration

In [ ]:
# Experiment definitions
experiments = {
    "exp1_rna_bg_noctr": {
        "type": "rna",
        "results_folder": "results/exp1_rna_bg_noctr/model",
        "label": "RNA: bg=T, ctr=OFF",
        "notebook": "model_comparisons/bone_marrow_gp_es_exp1_out.ipynb",
    },
    "exp2_rna_nobg_noctr": {
        "type": "rna",
        "results_folder": "results/exp2_rna_nobg_noctr/model",
        "label": "RNA: bg=F, ctr=OFF",
        "notebook": "model_comparisons/bone_marrow_gp_es_exp2_out.ipynb",
    },
    "exp3_rna_bg_ctr": {
        "type": "rna",
        "results_folder": "results/exp3_rna_bg_ctr/model",
        "label": "RNA: bg=T, ctr=ON",
        "notebook": "model_comparisons/bone_marrow_gp_es_exp3_out.ipynb",
    },
    "exp4_rna_nobg_ctr": {
        "type": "rna",
        "results_folder": "results/exp4_rna_nobg_ctr/model",
        "label": "RNA: bg=F, ctr=ON",
        "notebook": "model_comparisons/bone_marrow_gp_es_exp4_out.ipynb",
    },
    "exp5_mm_noctr": {
        "type": "multimodal",
        "results_folder": "results/exp5_mm_noctr/model",
        "label": "MM: ctr=OFF",
        "notebook": "bone_marrow_mm_es_exp5_out.ipynb",
    },
    "exp6_mm_ctr_rna": {
        "type": "multimodal",
        "results_folder": "results/exp6_mm_ctr_rna/model",
        "label": "MM: ctr=RNA",
        "notebook": "bone_marrow_mm_es_exp6_out.ipynb",
    },
    "exp7_mm_ctr_both": {
        "type": "multimodal",
        "results_folder": "results/exp7_mm_ctr_both/model",
        "label": "MM: ctr=RNA+ATAC",
        "notebook": "bone_marrow_mm_es_exp7_out.ipynb",
    },
}

# Check which experiments have completed
for name, cfg in experiments.items():
    model_path = cfg["results_folder"]
    exists = os.path.exists(model_path)
    print(f"{'OK' if exists else 'MISSING':>7}  {name:30s}  {model_path}")

## 2. Load model state dicts

In [ ]:
# Load state dicts (lighter than full model load — no adata needed)
state_dicts = {}
for name, cfg in experiments.items():
    model_path = cfg["results_folder"]
    ckpt_path = os.path.join(model_path, "model.pt")
    if not os.path.exists(ckpt_path):
        print(f"SKIP {name}: {ckpt_path} not found")
        continue
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dicts[name] = ckpt.get("model_state_dict", ckpt)
    print(f"Loaded {name}: {len(state_dicts[name])} params")

print(f"\n{len(state_dicts)}/{len(experiments)} experiments loaded")

## 3. Parameter summary tables

In [ ]:
# Key parameters to inspect
rna_params = [
    "px_r_mu",
    "px_r_log_sigma",
    "dispersion_prior_rate_raw",
    "additive_background",
    "feature_scaling",
    "library_log_means",
    "library_log_vars",
]

mm_params = [
    "px_r_mu.rna",
    "px_r_mu.atac",
    "px_r_log_sigma.rna",
    "px_r_log_sigma.atac",
    "dispersion_prior_rate_raw.rna",
    "dispersion_prior_rate_raw.atac",
    "additive_background.rna",
    "feature_scaling.rna",
    "feature_scaling.atac",
    "library_log_means_rna",
    "library_log_means_atac",
    "library_log_vars_rna",
    "library_log_vars_atac",
]


def param_stats(sd, param_keys):
    """Extract summary stats for specified parameters."""
    rows = []
    for key in param_keys:
        if key in sd:
            t = sd[key].float()
            rows.append(
                {
                    "param": key,
                    "shape": str(list(t.shape)),
                    "mean": t.mean().item(),
                    "std": t.std().item() if t.numel() > 1 else 0.0,
                    "min": t.min().item(),
                    "max": t.max().item(),
                }
            )
    return pd.DataFrame(rows)


for name, sd in state_dicts.items():
    cfg = experiments[name]
    keys = mm_params if cfg["type"] == "multimodal" else rna_params
    df = param_stats(sd, keys)
    if len(df) > 0:
        print(f"\n{'=' * 80}")
        print(f"{cfg['label']} ({name})")
        print(f"{'=' * 80}")
        print(df.to_string(index=False, float_format="{:.4f}".format))
    else:
        print(f"\n{name}: no matching params found")

## 4. Cross-experiment comparison

In [ ]:
# Compare key scalar metrics across experiments
def extract_scalars(sd, exp_type):
    """Extract key scalar summaries from state dict."""
    from torch.nn.functional import softplus

    row = {}
    if exp_type == "rna":
        if "px_r_mu" in sd:
            theta = torch.exp(sd["px_r_mu"].float())
            row["theta_median"] = theta.median().item()
            row["theta_mean"] = theta.mean().item()
            row["theta_max"] = theta.max().item()
        if "px_r_log_sigma" in sd:
            sigma = torch.exp(sd["px_r_log_sigma"].float())
            row["theta_sigma_mean"] = sigma.mean().item()
        if "dispersion_prior_rate_raw" in sd:
            rate = softplus(sd["dispersion_prior_rate_raw"].float())
            row["disp_rate_mean"] = rate.mean().item()
        if "library_log_means" in sd:
            row["lib_log_mean"] = sd["library_log_means"].float().mean().item()
        if "additive_background" in sd:
            bg = torch.exp(sd["additive_background"].float())
            row["bg_mean"] = bg.mean().item()
            row["bg_max"] = bg.max().item()
    else:
        for mod in ["rna", "atac"]:
            if f"px_r_mu.{mod}" in sd:
                theta = torch.exp(sd[f"px_r_mu.{mod}"].float())
                row[f"theta_{mod}_median"] = theta.median().item()
                row[f"theta_{mod}_mean"] = theta.mean().item()
            if f"px_r_log_sigma.{mod}" in sd:
                sigma = torch.exp(sd[f"px_r_log_sigma.{mod}"].float())
                row[f"theta_sigma_{mod}_mean"] = sigma.mean().item()
            if f"dispersion_prior_rate_raw.{mod}" in sd:
                rate = softplus(sd[f"dispersion_prior_rate_raw.{mod}"].float())
                row[f"disp_rate_{mod}_mean"] = rate.mean().item()
            if f"library_log_means_{mod}" in sd:
                row[f"lib_log_mean_{mod}"] = sd[f"library_log_means_{mod}"].float().mean().item()

    return row


comp_rows = []
for name, sd in state_dicts.items():
    cfg = experiments[name]
    row = {"experiment": cfg["label"]}
    row.update(extract_scalars(sd, cfg["type"]))
    comp_rows.append(row)

df_comp = pd.DataFrame(comp_rows).set_index("experiment")
print("Cross-experiment comparison:")
print(df_comp.to_string(float_format="{:.4f}".format))

## 5. Dispersion (theta) distributions

In [ ]:
# Plot theta = exp(px_r_mu) distributions across RNA experiments
rna_exps = {k: v for k, v in state_dicts.items() if experiments[k]["type"] == "rna"}

if rna_exps:
    fig, axes = plt.subplots(1, len(rna_exps), figsize=(5 * len(rna_exps), 4), sharey=True)
    if len(rna_exps) == 1:
        axes = [axes]
    for ax, (name, sd) in zip(axes, rna_exps.items(), strict=False):
        if "px_r_mu" in sd:
            theta = torch.exp(sd["px_r_mu"].float()).flatten().numpy()
            ax.hist(theta, bins=50, alpha=0.7, color="steelblue")
            ax.axvline(np.median(theta), color="red", ls="--", label=f"median={np.median(theta):.1f}")
            ax.set_xlabel("theta = exp(px_r_mu)")
            ax.set_title(experiments[name]["label"])
            ax.legend(fontsize=8)
    axes[0].set_ylabel("Count")
    fig.suptitle("Learned dispersion (theta) — RNA experiments", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot theta distributions for multimodal experiments
mm_exps = {k: v for k, v in state_dicts.items() if experiments[k]["type"] == "multimodal"}

if mm_exps:
    fig, axes = plt.subplots(2, len(mm_exps), figsize=(5 * len(mm_exps), 8), sharey="row")
    if len(mm_exps) == 1:
        axes = axes.reshape(-1, 1)
    for j, (name, sd) in enumerate(mm_exps.items()):
        for i, mod in enumerate(["rna", "atac"]):
            key = f"px_r_mu.{mod}"
            if key in sd:
                theta = torch.exp(sd[key].float()).flatten().numpy()
                axes[i, j].hist(theta, bins=50, alpha=0.7, color="steelblue" if mod == "rna" else "darkorange")
                axes[i, j].axvline(np.median(theta), color="red", ls="--", label=f"median={np.median(theta):.1f}")
                axes[i, j].set_xlabel(f"theta_{mod}")
                axes[i, j].legend(fontsize=8)
            if i == 0:
                axes[i, j].set_title(experiments[name]["label"])
        axes[0, j].set_ylabel("RNA")
        axes[1, j].set_ylabel("ATAC")
    fig.suptitle("Learned dispersion (theta) — Multimodal experiments", fontsize=14)
    plt.tight_layout()
    plt.show()

## 6. Dispersion posterior uncertainty (sigma)

In [ ]:
# Scatter: theta_mu vs theta_sigma for each experiment
all_exps = list(state_dicts.items())
n = len(all_exps)
if n > 0:
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4), sharey=True)
    if n == 1:
        axes = [axes]
    for ax, (name, sd) in zip(axes, all_exps, strict=False):
        cfg = experiments[name]
        if cfg["type"] == "rna":
            mu_key, sig_key = "px_r_mu", "px_r_log_sigma"
        else:
            mu_key, sig_key = "px_r_mu.rna", "px_r_log_sigma.rna"
        if mu_key in sd and sig_key in sd:
            mu = sd[mu_key].float().flatten().numpy()
            sigma = torch.exp(sd[sig_key].float()).flatten().numpy()
            ax.scatter(mu, sigma, alpha=0.3, s=3)
            ax.set_xlabel("px_r_mu (log theta)")
            ax.set_title(cfg["label"], fontsize=10)
    axes[0].set_ylabel("px_r_sigma")
    fig.suptitle("Dispersion: mean vs posterior std", fontsize=14)
    plt.tight_layout()
    plt.show()

## 7. Library size prior comparison

In [ ]:
# Compare library_log_means across experiments
print("Library log-means per experiment:")
print(f"{'Experiment':<30s} {'mean':>8s} {'std':>8s} {'min':>8s} {'max':>8s}")
print("-" * 70)
for name, sd in state_dicts.items():
    cfg = experiments[name]
    if cfg["type"] == "rna":
        key = "library_log_means"
    else:
        key = "library_log_means_rna"
    if key in sd:
        t = sd[key].float()
        print(
            f"{cfg['label']:<30s} {t.mean().item():>8.4f} {t.std().item():>8.4f} {t.min().item():>8.4f} {t.max().item():>8.4f}"
        )

print()
print("Expected: centered experiments should have mean ~0, uncentered ~7.5")

## 8. Training history comparison

Load training history from output notebooks (if available) to compare ELBO convergence.

In [ ]:
# Load training history from model checkpoints
histories = {}
for name, cfg in experiments.items():
    model_path = cfg["results_folder"]
    attr_path = os.path.join(model_path, "attr.pkl")
    if os.path.exists(attr_path):
        import pickle

        with open(attr_path, "rb") as f:
            attrs = pickle.load(f)
        if "history_" in attrs:
            histories[name] = attrs["history_"]
            print(f"{name}: {len(attrs['history_'].get('elbo_train', []))} epochs")

if histories:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for name, hist in histories.items():
        label = experiments[name]["label"]
        if "elbo_train" in hist:
            train_elbo = hist["elbo_train"]["elbo_train"].values
            axes[0].plot(train_elbo, label=label, alpha=0.8)
        if "elbo_validation" in hist:
            val_elbo = hist["elbo_validation"]["elbo_validation"].values
            axes[1].plot(val_elbo, label=label, alpha=0.8)
    axes[0].set_title("Train ELBO")
    axes[1].set_title("Validation ELBO")
    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.set_ylabel("ELBO")
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("No training histories found yet.")

## 9. Interpretation

Key questions to answer from the results above:

1. **Does variational px_r prevent theta explosion?**
   - Target: median theta in 5-50 range, not 400+ as seen with MAP optimisation.
   - Check the theta histograms (Section 5) and cross-experiment table (Section 4).
   - The posterior sigma (Section 6) should be non-negligible — if sigma collapses to ~0 the variational posterior is behaving like MAP.

2. **Does library centering speed up convergence?**
   - Compare training curves (Section 8) between centered (exp3/4) and uncentered (exp1/2) RNA experiments.
   - Centered experiments should converge in fewer epochs because the encoder starts near the prior (mean ~0 vs ~7.5).
   - Check final ELBO values — centering should not hurt final performance.

3. **Does regularise_background=False hurt or help?**
   - Compare exp1 (bg=T, ctr=OFF) vs exp2 (bg=F, ctr=OFF) and exp3 vs exp4 (same with centering ON).
   - With the previous Gamma(1,100) prior, background was crushed to ~0.00004 — effectively disabled.
   - If bg=False matches or beats bg=True, the background prior was not contributing.

4. **How do multimodal centering variants compare?**
   - exp5 (no centering) vs exp6 (RNA only) vs exp7 (RNA + ATAC).
   - Does centering RNA library help the joint latent?
   - Does additionally centering ATAC (sensitivity=0.2) help or hurt?